# Independent Tariff, Holiday and Weather Checks

## Research question

> Were household electricity-consumption patterns associated with implemented
> tariff revisions, official public holidays, and weather conditions?

This notebook is independent of the household clustering analysis.

It does **not** use cluster labels, archetype names, cluster-specific
comparisons, clustering features, or cluster membership as a control variable.
It does not modify the verified EDA or clustering notebooks.

The analyses are contextual and associational. They do not establish causal
tariff, holiday, or weather effects.

## Analysis design

- **Tariffs:** PUCSL-approved and implemented effective dates only. Proposal,
  consultation, requested-percentage and announcement-only records are rejected.
- **Primary tariff source:** six-hour cumulative registers reconstructed to
  household-day estimates.
- **High-resolution tariff check:** strict 15-minute days for the 16 July 2024
  revision.
- **Holidays:** official public holidays only. Each holiday is compared with
  nearby non-holiday controls having the same weekday and tariff regime.
- **Weather:** NASA POWER daily data. Returned coordinates and complete weather
  series are audited to detect branches sharing the same coarse grid data.
- **December 2024:** included in the 15-minute contextual period only when the
  prespecified completeness and non-truncation checks pass.

## Full post-run audit correction

The first executed run stopped during six-hour daily reconstruction because
households without a recognized branch received a missing `branch_conflict`
value. Pandas represented that column as floating point, so applying the
boolean inversion operator caused:

`TypeError: bad operand type for unary ~: 'float'`

This release corrects that integration error and also makes two methodological
corrections identified during the full notebook audit:

1. the paired effect size is now the Wilcoxon matched-pairs rank-biserial
   correlation based on signed-rank sums, rather than a simple positive-versus-
   negative sign count;
2. weather-consumption analysis is performed at `weather_series_id`–date level.
   Branches with identical complete NASA POWER series are combined into one
   regional weather observation instead of being treated as distinct local
   weather measurements.

The tariff table remains restricted to PUCSL-approved implemented regimes, the
holiday table remains restricted to official public holidays, and no clustering
inputs are loaded.

## Coverage-corrected six-hour method

The earlier 8-hour maximum-gap rule was too sparse for household-day event analysis. The contextual notebook now uses a maximum cumulative-register interval of 24 hours with at least 18 covered hours per household-day. This threshold is accepted only when the reconstructed monthly totals independently pass a billing-validation gate (at least 1,000 matched household-months at >=75% coverage, Spearman >=0.85, median APE <=20%, and exact allocation conservation). The analysis remains associational and independent from clustering.

In [ ]:
# ============================================================
# STEP 0 — PACKAGE CHECK
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
import hashlib
import json
import os
import re
import warnings

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata as scipy_rankdata
from scipy.stats import spearmanr, wilcoxon
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

print("Core packages imported successfully.")

In [ ]:
# ============================================================
# STEP 1 — CONFIGURATION AND SEPARATION FROM CLUSTERING
# ============================================================

PACKAGE_ROOT = Path.cwd().resolve()

if PACKAGE_ROOT.name != "INDEPENDENT_CONTEXTUAL_CHECKS":
    candidate = PACKAGE_ROOT / "INDEPENDENT_CONTEXTUAL_CHECKS"
    if candidate.exists():
        PACKAGE_ROOT = candidate.resolve()

PROJECT_ROOT = Path(
    os.environ.get(
        "COMPLETE_EDA_ROOT",
        PACKAGE_ROOT.parent,
    )
).expanduser().resolve()

OUTPUT_DIR = PACKAGE_ROOT / "outputs"
SOURCE_DIR = PACKAGE_ROOT / "sources"
RAW_WEATHER_DIR = SOURCE_DIR / "nasa_power_raw"

for directory in [
    OUTPUT_DIR,
    SOURCE_DIR,
    RAW_WEATHER_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

TARIFF_PATH = (
    PACKAGE_ROOT
    / "official_tariff_regimes.csv"
)
HOLIDAY_PATH = (
    PACKAGE_ROOT
    / "official_sri_lanka_holidays.csv"
)
WEATHER_PATH = (
    PACKAGE_ROOT
    / "nasa_power_daily_weather.csv"
)
COORDINATE_PATH = (
    SOURCE_DIR
    / "branch_coordinates.csv"
)

SMART15_DAILY_DIR = (
    PROJECT_ROOT
    / "complete_eda_safe_outputs"
    / "smart15_work"
    / "daily"
)

SIXHOUR_PATTERN = "smart_6hour_*.csv"
MONTHLY_PATH = PROJECT_ROOT / "monthly_consumption.csv"

SMART15_EXPECTED_PARTITIONS = 64
SMART15_STRICT_INTERVALS = 90
SMART15_MIN_STRICT_DAYS = 30

SIXHOUR_MAX_GAP_HOURS = 24.0
SIXHOUR_PRIMARY_MIN_COVERED_HOURS = 18.0
SIXHOUR_SENSITIVITY_MIN_COVERED_HOURS = 12.0
SIXHOUR_VALIDATION_MIN_MATCHED_MONTHS = 1000
SIXHOUR_VALIDATION_MIN_SPEARMAN = 0.85
SIXHOUR_VALIDATION_MAX_MEDIAN_APE = 0.20
SIXHOUR_MONTHLY_MIN_COVERAGE = 0.75
MAX_RATE_KWH_PER_HOUR = 24.0

EVENT_WINDOW_DAYS = 28
MIN_EVENT_HOUSEHOLDS = 30
MIN_EVENT_DATE_PAIRS = 2
MAX_HOLIDAY_CONTROLS = 4
RANDOM_SEED = 42

WEATHER_START = "20230101"
WEATHER_END = "20241231"
WEATHER_PARAMETERS = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",
    "PRECTOTCORR",
    "WS2M",
]

FORBIDDEN_CLUSTER_TERMS = [
    "final_cluster",
    "cluster_label",
    "cluster_name",
    "archetype",
    "kmeans",
    "cluster_membership",
]

print("Package root:", PACKAGE_ROOT)
print("Project root:", PROJECT_ROOT)
print("This notebook does not load or modify clustering outputs.")

In [ ]:
# ============================================================
# STEP 2 — HELPER FUNCTIONS
# ============================================================

def clean_id(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def clean_numeric(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce",
    )


def parse_explicit_datetime(
    frame,
    date_column,
    time_column=None,
):
    if time_column is None:
        text = (
            frame[date_column]
            .astype(str)
            .str.strip()
        )
    else:
        text = (
            frame[date_column]
            .astype(str)
            .str.strip()
            + " "
            + frame[time_column]
            .astype(str)
            .str.strip()
        )

    parsed = pd.to_datetime(
        text,
        errors="coerce",
    )

    if parsed.notna().sum() == 0:
        parsed = pd.to_datetime(
            text,
            errors="coerce",
            dayfirst=True,
        )

    return parsed


def rank_biserial_from_paired(differences):
    """
    Wilcoxon matched-pairs rank-biserial correlation.

    Positive values indicate that the second/after condition tends
    to be higher; negative values indicate that it tends to be lower.
    Zero differences are removed, matching the Wilcoxon convention.
    """
    values = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    values = values.loc[
        values.ne(0)
    ]

    if values.empty:
        return np.nan

    absolute_ranks = pd.Series(
        scipy_rankdata(
            values.abs(),
            method="average",
        ),
        index=values.index,
    )

    positive_rank_sum = float(
        absolute_ranks.loc[
            values.gt(0)
        ].sum()
    )
    negative_rank_sum = float(
        absolute_ranks.loc[
            values.lt(0)
        ].sum()
    )

    denominator = (
        positive_rank_sum
        + negative_rank_sum
    )

    if denominator == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / denominator


def safe_wilcoxon(differences):
    values = pd.Series(differences).dropna()

    if len(values) < 10:
        return np.nan, np.nan

    if values.eq(0).all():
        return 0.0, 1.0

    result = wilcoxon(values)

    return float(result.statistic), float(result.pvalue)


def adjust_pvalues(values, method="holm"):
    series = pd.Series(values)
    valid = series.notna()
    adjusted = pd.Series(
        np.nan,
        index=series.index,
        dtype=float,
    )

    if valid.any():
        adjusted.loc[valid] = multipletests(
            series.loc[valid],
            method=method,
        )[1]

    return adjusted


def canonical_branch(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().upper()

    for branch in [
        "MORATUWA",
        "NUGEGODA",
        "KELANIYA",
        "KALUTARA",
    ]:
        if branch in text:
            return branch

    return np.nan


def save_figure(filename):
    path = OUTPUT_DIR / filename
    plt.tight_layout()
    plt.savefig(
        path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close()


def assign_tariff_regime(
    dates,
    regimes,
):
    date_series = pd.to_datetime(
        dates,
        errors="coerce",
    )

    labels = pd.Series(
        pd.NA,
        index=date_series.index,
        dtype="string",
    )

    for row in regimes.itertuples():
        start = pd.Timestamp(
            row.effective_start
        )
        end = (
            pd.Timestamp(row.effective_end)
            if pd.notna(row.effective_end)
            else pd.Timestamp.max.normalize()
        )

        mask = date_series.between(
            start,
            end,
            inclusive="both",
        )
        labels.loc[mask] = row.regime_id

    return labels


def weather_series_hash(group):
    columns = [
        "date",
        "temperature_mean_c",
        "temperature_max_c",
        "temperature_min_c",
        "relative_humidity_percent",
        "precipitation_corrected_mm_day",
        "wind_speed_2m_m_s",
    ]

    payload = (
        group.sort_values("date")[columns]
        .round(6)
        .to_csv(index=False)
        .encode("utf-8")
    )

    return hashlib.sha256(
        payload
    ).hexdigest()[:16]


def weather_distance(
    left_temperature,
    left_rain,
    right_temperature,
    right_rain,
    temperature_scale,
    rain_scale,
):
    return (
        abs(
            left_temperature
            - right_temperature
        )
        / temperature_scale
        + abs(
            np.log1p(left_rain)
            - np.log1p(right_rain)
        )
        / rain_scale
    )


def build_weather_matched_tariff_pairs(
    weather,
    event_date,
    holiday_dates,
    window_days=28,
):
    event_date = pd.Timestamp(event_date)

    pre_start = (
        event_date
        - pd.Timedelta(days=window_days)
    )
    pre_end = (
        event_date
        - pd.Timedelta(days=1)
    )
    post_start = (
        event_date
        + pd.Timedelta(days=1)
    )
    post_end = (
        event_date
        + pd.Timedelta(days=window_days)
    )

    weather_base = (
        weather.drop_duplicates(
            [
                "weather_series_id",
                "date",
            ]
        )
        .copy()
    )

    weather_base = weather_base.loc[
        ~weather_base["date"].isin(
            holiday_dates
        )
    ].copy()

    pair_rows = []

    for series_id, series in weather_base.groupby(
        "weather_series_id"
    ):
        pre = series.loc[
            series["date"].between(
                pre_start,
                pre_end,
            )
        ].copy()

        post = series.loc[
            series["date"].between(
                post_start,
                post_end,
            )
        ].copy()

        for weekday in range(7):
            pre_day = pre.loc[
                pre["date"].dt.dayofweek.eq(
                    weekday
                )
            ].copy()

            post_day = post.loc[
                post["date"].dt.dayofweek.eq(
                    weekday
                )
            ].copy()

            if pre_day.empty or post_day.empty:
                continue

            combined = pd.concat(
                [pre_day, post_day],
                ignore_index=True,
            )

            temperature_scale = max(
                float(
                    combined[
                        "temperature_mean_c"
                    ].std(ddof=0)
                ),
                0.5,
            )

            rain_scale = max(
                float(
                    np.log1p(
                        combined[
                            "precipitation_corrected_mm_day"
                        ].clip(lower=0)
                    ).std(ddof=0)
                ),
                0.25,
            )

            cost = np.zeros(
                (
                    len(pre_day),
                    len(post_day),
                )
            )

            for i, pre_row in enumerate(
                pre_day.itertuples()
            ):
                for j, post_row in enumerate(
                    post_day.itertuples()
                ):
                    cost[i, j] = weather_distance(
                        pre_row.temperature_mean_c,
                        pre_row.precipitation_corrected_mm_day,
                        post_row.temperature_mean_c,
                        post_row.precipitation_corrected_mm_day,
                        temperature_scale,
                        rain_scale,
                    )

            pre_index, post_index = (
                linear_sum_assignment(cost)
            )

            for i, j in zip(
                pre_index,
                post_index,
            ):
                pair_rows.append({
                    "weather_series_id": series_id,
                    "weekday_number": weekday,
                    "pre_date": (
                        pre_day.iloc[i]["date"]
                    ),
                    "post_date": (
                        post_day.iloc[j]["date"]
                    ),
                    "weather_distance": (
                        float(cost[i, j])
                    ),
                })

    pairs = pd.DataFrame(pair_rows)

    if not pairs.empty:
        pairs["pair_id"] = (
            np.arange(len(pairs))
        )

    return pairs


def evaluate_date_pairs(
    daily,
    pairs,
    outcome_column,
    event_id,
    source_name,
):
    if pairs.empty:
        return (
            pd.DataFrame(),
            pd.DataFrame([{
                "event_id": event_id,
                "source": source_name,
                "status": "no_weather_matched_date_pairs",
            }]),
        )

    required = [
        "household_ID",
        "analysis_date",
        "weather_series_id",
        outcome_column,
    ]

    daily_base = daily[
        required
    ].dropna().copy()

    pre = pairs.merge(
        daily_base,
        left_on=[
            "weather_series_id",
            "pre_date",
        ],
        right_on=[
            "weather_series_id",
            "analysis_date",
        ],
        how="inner",
    ).rename(
        columns={
            outcome_column: "pre_kwh",
        }
    )

    post = pairs.merge(
        daily_base,
        left_on=[
            "weather_series_id",
            "post_date",
        ],
        right_on=[
            "weather_series_id",
            "analysis_date",
        ],
        how="inner",
    ).rename(
        columns={
            outcome_column: "post_kwh",
        }
    )

    matched = pre[
        [
            "pair_id",
            "household_ID",
            "pre_kwh",
        ]
    ].merge(
        post[
            [
                "pair_id",
                "household_ID",
                "post_kwh",
            ]
        ],
        on=[
            "pair_id",
            "household_ID",
        ],
        how="inner",
        validate="one_to_one",
    )

    matched["difference_kwh"] = (
        matched["post_kwh"]
        - matched["pre_kwh"]
    )

    matched["percent_change"] = (
        matched["difference_kwh"]
        / matched["pre_kwh"].replace(
            0,
            np.nan,
        )
        * 100
    )

    household_effects = (
        matched.groupby(
            "household_ID",
            as_index=False,
        )
        .agg(
            matched_date_pairs=(
                "pair_id",
                "nunique",
            ),
            pre_kwh=("pre_kwh", "mean"),
            post_kwh=("post_kwh", "mean"),
            difference_kwh=(
                "difference_kwh",
                "mean",
            ),
            percent_change=(
                "percent_change",
                "median",
            ),
        )
    )

    household_effects = (
        household_effects.loc[
            household_effects[
                "matched_date_pairs"
            ] >= MIN_EVENT_DATE_PAIRS
        ]
        .copy()
    )

    statistic, p_value = safe_wilcoxon(
        household_effects[
            "difference_kwh"
        ]
    )

    status = (
        "completed"
        if len(household_effects)
        >= MIN_EVENT_HOUSEHOLDS
        else "insufficient_households"
    )

    summary = pd.DataFrame([{
        "event_id": event_id,
        "source": source_name,
        "status": status,
        "households": len(
            household_effects
        ),
        "date_pairs": int(
            pairs["pair_id"].nunique()
        ),
        "median_pre_kwh": (
            household_effects[
                "pre_kwh"
            ].median()
        ),
        "median_post_kwh": (
            household_effects[
                "post_kwh"
            ].median()
        ),
        "median_difference_kwh": (
            household_effects[
                "difference_kwh"
            ].median()
        ),
        "median_percent_change": (
            household_effects[
                "percent_change"
            ].median()
        ),
        "wilcoxon_statistic": statistic,
        "p_value": p_value,
        "rank_biserial": (
            rank_biserial_from_paired(
                household_effects[
                    "difference_kwh"
                ]
            )
        ),
        "interpretation": (
            "Consumption change associated with the implemented "
            "tariff revision; this is not a causal estimate."
        ),
    }])

    household_effects[
        "event_id"
    ] = event_id
    household_effects["source"] = (
        source_name
    )

    return household_effects, summary

In [ ]:
# ============================================================
# STEP 3 — VERIFY EXTERNAL SOURCE TABLES
# ============================================================

tariffs = pd.read_csv(
    TARIFF_PATH
)
holidays = pd.read_csv(
    HOLIDAY_PATH
)
coordinates = pd.read_csv(
    COORDINATE_PATH
)

tariffs["effective_start"] = (
    pd.to_datetime(
        tariffs["effective_start"],
        errors="raise",
    )
)
tariffs["effective_end"] = (
    pd.to_datetime(
        tariffs["effective_end"],
        errors="coerce",
    )
)
tariffs["decision_date"] = (
    pd.to_datetime(
        tariffs["decision_date"],
        errors="raise",
    )
)

holidays["date"] = pd.to_datetime(
    holidays["date"],
    errors="raise",
)


for boolean_column in [
    "is_public_holiday",
    "is_special_declaration",
]:
    normalized_boolean = (
        holidays[boolean_column]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "1": True,
            "yes": True,
            "false": False,
            "0": False,
            "no": False,
        })
    )

    if normalized_boolean.isna().any():
        raise RuntimeError(
            f"Holiday source gate failed: invalid boolean "
            f"values in {boolean_column}."
        )

    holidays[boolean_column] = (
        normalized_boolean.astype(bool)
    )


expected_tariff_periods = {
    "tariff_2023_feb": (
        pd.Timestamp("2023-02-15"),
        pd.Timestamp("2023-06-30"),
    ),
    "tariff_2023_jul": (
        pd.Timestamp("2023-07-01"),
        pd.Timestamp("2023-10-19"),
    ),
    "tariff_2023_oct": (
        pd.Timestamp("2023-10-20"),
        pd.Timestamp("2024-03-04"),
    ),
    "tariff_2024_mar": (
        pd.Timestamp("2024-03-05"),
        pd.Timestamp("2024-07-15"),
    ),
    "tariff_2024_jul": (
        pd.Timestamp("2024-07-16"),
        pd.NaT,
    ),
}

actual_tariff_periods = {
    row.regime_id: (
        pd.Timestamp(row.effective_start),
        (
            pd.Timestamp(row.effective_end)
            if pd.notna(row.effective_end)
            else pd.NaT
        ),
    )
    for row in tariffs.itertuples()
}

if set(actual_tariff_periods) != set(
    expected_tariff_periods
):
    raise RuntimeError(
        "Tariff source gate failed: the implemented regime IDs "
        "do not match the five verified PUCSL regimes."
    )

for regime_id, (
    expected_start,
    expected_end,
) in expected_tariff_periods.items():
    actual_start, actual_end = (
        actual_tariff_periods[regime_id]
    )

    end_matches = (
        pd.isna(expected_end)
        and pd.isna(actual_end)
    ) or (
        pd.notna(expected_end)
        and pd.notna(actual_end)
        and actual_end == expected_end
    )

    if (
        actual_start != expected_start
        or not end_matches
    ):
        raise RuntimeError(
            "Tariff source gate failed: unexpected implemented "
            f"period for {regime_id}."
        )

ordered_tariffs = tariffs.sort_values(
    "effective_start"
).reset_index(drop=True)

for position in range(
    len(ordered_tariffs) - 1
):
    current_end = ordered_tariffs.loc[
        position,
        "effective_end",
    ]
    next_start = ordered_tariffs.loc[
        position + 1,
        "effective_start",
    ]

    if (
        pd.isna(current_end)
        or current_end
        + pd.Timedelta(days=1)
        != next_start
    ):
        raise RuntimeError(
            "Tariff source gate failed: implemented tariff "
            "periods contain a gap or overlap."
        )

if holidays["date"].duplicated().any():
    raise RuntimeError(
        "Holiday source gate failed: duplicate public-holiday dates."
    )

if not holidays["date"].dt.year.isin(
    [2023, 2024]
).all():
    raise RuntimeError(
        "Holiday source gate failed: dates outside 2023-2024."
    )

if not tariffs[
    "decision_status"
].eq(
    "approved_implemented"
).all():
    raise RuntimeError(
        "Tariff source gate failed: "
        "every row must be approved and implemented."
    )

rejected_words = re.compile(
    r"proposed|consultation|request",
    flags=re.IGNORECASE,
)

if tariffs[
    "source_document"
].fillna("").str.contains(
    rejected_words
).any():
    raise RuntimeError(
        "Tariff source gate failed: "
        "a proposal or consultation record entered the regime table."
    )

if not tariffs[
    "source_url"
].str.contains(
    "pucsl.gov.lk",
    case=False,
).all():
    raise RuntimeError(
        "Tariff source gate failed: "
        "all decision sources must be PUCSL."
    )

if not holidays[
    "is_public_holiday"
].astype(bool).all():
    raise RuntimeError(
        "Holiday source gate failed: "
        "non-public holidays are present."
    )

allowed_holiday_domains = (
    holidays["source_url"]
    .str.contains(
        r"cbsl\.gov\.lk|documents\.gov\.lk",
        case=False,
        regex=True,
    )
)

if not allowed_holiday_domains.all():
    raise RuntimeError(
        "Holiday source gate failed: "
        "all records must come from official government or CBSL sources."
    )

if coordinates["branch"].duplicated().any():
    raise RuntimeError(
        "Branch coordinate table contains duplicate branches."
    )

display(tariffs)
display(
    holidays.groupby(
        [
            holidays["date"].dt.year,
            "holiday_category",
        ]
    ).size().rename("holidays")
)
display(coordinates)

tariffs.to_csv(
    OUTPUT_DIR
    / "01_verified_implemented_tariff_regimes.csv",
    index=False,
)
holidays.to_csv(
    OUTPUT_DIR
    / "02_verified_official_public_holidays.csv",
    index=False,
)

In [ ]:
# ============================================================
# STEP 4 — NASA POWER DAILY WEATHER DOWNLOAD AND GRID AUDIT
# ============================================================

NASA_BASE_URL = (
    "https://power.larc.nasa.gov/"
    "api/temporal/daily/point"
)

weather_frames = []
weather_request_rows = []

for row in coordinates.itertuples():
    parameters = ",".join(
        WEATHER_PARAMETERS
    )

    request_params = {
        "parameters": parameters,
        "community": "RE",
        "longitude": float(row.longitude),
        "latitude": float(row.latitude),
        "start": WEATHER_START,
        "end": WEATHER_END,
        "format": "JSON",
        "time-standard": "LST",
    }

    api_url = requests.Request(
        "GET",
        NASA_BASE_URL,
        params=request_params,
    ).prepare().url

    raw_path = (
        RAW_WEATHER_DIR
        / f"{row.branch.lower()}_2023_2024.json"
    )

    content = None
    download_error = None

    try:
        response = requests.get(
            NASA_BASE_URL,
            params=request_params,
            timeout=180,
        )
        response.raise_for_status()
        content = response.json()

        with raw_path.open(
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                content,
                handle,
                indent=2,
            )

    except Exception as error:
        download_error = str(error)

        if raw_path.exists():
            with raw_path.open(
                "r",
                encoding="utf-8",
            ) as handle:
                content = json.load(handle)

    if content is None:
        raise RuntimeError(
            f"NASA POWER download failed for {row.branch}: "
            f"{download_error}"
        )

    if (
        "properties" not in content
        or "parameter" not in content["properties"]
    ):
        raise RuntimeError(
            f"Unexpected NASA POWER response for {row.branch}."
        )

    parameters_data = (
        content["properties"]["parameter"]
    )

    returned_coordinates = (
        content.get(
            "geometry",
            {},
        ).get(
            "coordinates",
            [np.nan, np.nan, np.nan],
        )
    )

    all_dates = sorted(
        set().union(
            *[
                set(
                    parameters_data[
                        parameter
                    ].keys()
                )
                for parameter in WEATHER_PARAMETERS
            ]
        )
    )

    records = []

    for date_key in all_dates:
        values = {}

        for parameter in WEATHER_PARAMETERS:
            value = parameters_data[
                parameter
            ].get(
                date_key,
                np.nan,
            )

            values[parameter] = (
                np.nan
                if value == -999
                else value
            )

        records.append({
            "branch": row.branch,
            "date": pd.to_datetime(
                date_key,
                format="%Y%m%d",
            ),
            "request_latitude": float(
                row.latitude
            ),
            "request_longitude": float(
                row.longitude
            ),
            "returned_longitude": (
                returned_coordinates[0]
                if len(returned_coordinates) > 0
                else np.nan
            ),
            "returned_latitude": (
                returned_coordinates[1]
                if len(returned_coordinates) > 1
                else np.nan
            ),
            "elevation_m": (
                returned_coordinates[2]
                if len(returned_coordinates) > 2
                else np.nan
            ),
            "temperature_mean_c": values[
                "T2M"
            ],
            "temperature_max_c": values[
                "T2M_MAX"
            ],
            "temperature_min_c": values[
                "T2M_MIN"
            ],
            "relative_humidity_percent": values[
                "RH2M"
            ],
            "precipitation_corrected_mm_day": values[
                "PRECTOTCORR"
            ],
            "wind_speed_2m_m_s": values[
                "WS2M"
            ],
            "api_url": api_url,
            "retrieved_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        })

    branch_weather = pd.DataFrame(
        records
    )

    branch_weather[
        "weather_series_id"
    ] = weather_series_hash(
        branch_weather
    )

    weather_frames.append(
        branch_weather
    )

    weather_request_rows.append({
        "branch": row.branch,
        "request_latitude": row.latitude,
        "request_longitude": row.longitude,
        "returned_longitude": (
            returned_coordinates[0]
            if len(returned_coordinates) > 0
            else np.nan
        ),
        "returned_latitude": (
            returned_coordinates[1]
            if len(returned_coordinates) > 1
            else np.nan
        ),
        "elevation_m": (
            returned_coordinates[2]
            if len(returned_coordinates) > 2
            else np.nan
        ),
        "weather_series_id": (
            branch_weather[
                "weather_series_id"
            ].iloc[0]
        ),
        "records": len(branch_weather),
        "download_error_if_cache_used": (
            download_error
        ),
        "api_url": api_url,
    })

weather = pd.concat(
    weather_frames,
    ignore_index=True,
)

weather["date"] = pd.to_datetime(
    weather["date"]
)

weather = weather.sort_values(
    [
        "branch",
        "date",
    ]
).reset_index(drop=True)

weather_request_audit = pd.DataFrame(
    weather_request_rows
)

weather_grid_audit = pd.DataFrame([{
    "branches_requested": (
        weather["branch"].nunique()
    ),
    "unique_returned_coordinate_pairs": (
        weather[
            [
                "returned_latitude",
                "returned_longitude",
            ]
        ].drop_duplicates().shape[0]
    ),
    "unique_complete_weather_series": (
        weather[
            "weather_series_id"
        ].nunique()
    ),
    "date_start": weather["date"].min(),
    "date_end": weather["date"].max(),
    "missing_temperature": (
        weather[
            "temperature_mean_c"
        ].isna().sum()
    ),
    "missing_precipitation": (
        weather[
            "precipitation_corrected_mm_day"
        ].isna().sum()
    ),
}])


expected_weather_dates = pd.date_range(
    "2023-01-01",
    "2024-12-31",
    freq="D",
)

for branch, branch_data in weather.groupby(
    "branch"
):
    observed_dates = pd.DatetimeIndex(
        branch_data["date"].sort_values()
    )

    if not observed_dates.equals(
        expected_weather_dates
    ):
        raise RuntimeError(
            f"NASA POWER gate failed for {branch}: "
            "the daily date sequence is incomplete or duplicated."
        )

    if branch_data[
        [
            "temperature_mean_c",
            "temperature_max_c",
            "temperature_min_c",
            "relative_humidity_percent",
            "precipitation_corrected_mm_day",
            "wind_speed_2m_m_s",
        ]
    ].isna().any().any():
        raise RuntimeError(
            f"NASA POWER gate failed for {branch}: "
            "required weather values are missing."
        )

weather.to_csv(
    WEATHER_PATH,
    index=False,
)
weather_request_audit.to_csv(
    OUTPUT_DIR
    / "03_nasa_power_request_and_grid_audit.csv",
    index=False,
)
weather_grid_audit.to_csv(
    OUTPUT_DIR
    / "04_nasa_power_grid_summary.csv",
    index=False,
)

display(weather_request_audit)
display(weather_grid_audit)

if weather[
    "weather_series_id"
].nunique() < weather[
    "branch"
].nunique():
    print(
        "Some branches share an identical complete NASA POWER "
        "weather series. Those branches will use one shared "
        "weather_series_id rather than being treated as distinct "
        "local weather observations."
    )

In [ ]:
# ============================================================
# STEP 5 — DECEMBER 2024 COVERAGE AUDIT
# ============================================================

daily_files = sorted(
    SMART15_DAILY_DIR.glob(
        "daily_*.parquet"
    )
)

if len(daily_files) != (
    SMART15_EXPECTED_PARTITIONS
):
    raise RuntimeError(
        "Expected "
        f"{SMART15_EXPECTED_PARTITIONS} "
        "15-minute daily Parquet partitions, "
        f"found {len(daily_files)}."
    )

daily_pattern = str(
    SMART15_DAILY_DIR
    / "daily_*.parquet"
).replace("\\", "/")

connection = duckdb.connect(
    database=":memory:"
)

monthly_coverage = connection.execute(f"""
WITH daily AS (
    SELECT
        household_ID,
        CAST(reading_date AS DATE)
            AS reading_date,
        valid_import_intervals
    FROM read_parquet('{daily_pattern}')
    WHERE reading_date >= DATE '2024-05-01'
      AND reading_date < DATE '2025-01-01'
),
daily_summary AS (
    SELECT
        reading_date,
        COUNT(DISTINCT household_ID)
            AS households_any_data,
        COUNT(DISTINCT CASE
            WHEN valid_import_intervals >= 72
            THEN household_ID
        END) AS households_at_least_72,
        COUNT(DISTINCT CASE
            WHEN valid_import_intervals >= 90
            THEN household_ID
        END) AS households_at_least_90,
        MEDIAN(valid_import_intervals)
            AS median_valid_intervals
    FROM daily
    GROUP BY reading_date
)
SELECT
    DATE_TRUNC('month', reading_date)
        AS month_start,
    COUNT(DISTINCT reading_date)
        AS observed_calendar_dates,
    MEDIAN(households_any_data)
        AS median_households_any_data,
    MEDIAN(households_at_least_72)
        AS median_households_at_least_72,
    MEDIAN(households_at_least_90)
        AS median_households_at_least_90,
    MEDIAN(median_valid_intervals)
        AS median_daily_interval_count,
    MIN(reading_date) AS first_observed_date,
    MAX(reading_date) AS last_observed_date
FROM daily_summary
GROUP BY DATE_TRUNC('month', reading_date)
ORDER BY month_start
""").df()

december_daily = connection.execute(f"""
SELECT
    CAST(reading_date AS DATE)
        AS reading_date,
    COUNT(DISTINCT household_ID)
        AS households_any_data,
    COUNT(DISTINCT CASE
        WHEN valid_import_intervals >= 72
        THEN household_ID
    END) AS households_at_least_72,
    COUNT(DISTINCT CASE
        WHEN valid_import_intervals >= 90
        THEN household_ID
    END) AS households_at_least_90,
    MEDIAN(valid_import_intervals)
        AS median_valid_intervals
FROM read_parquet('{daily_pattern}')
WHERE reading_date >= DATE '2024-12-01'
  AND reading_date < DATE '2025-01-01'
GROUP BY reading_date
ORDER BY reading_date
""").df()

connection.close()

monthly_coverage["month_start"] = (
    pd.to_datetime(
        monthly_coverage["month_start"]
    )
)
december_daily["reading_date"] = (
    pd.to_datetime(
        december_daily["reading_date"]
    )
)

reference = monthly_coverage.loc[
    monthly_coverage[
        "month_start"
    ].between(
        "2024-05-01",
        "2024-11-01",
    )
]

december = monthly_coverage.loc[
    monthly_coverage[
        "month_start"
    ].eq(
        pd.Timestamp(
            "2024-12-01"
        )
    )
]

if len(december) != 1:
    raise RuntimeError(
        "December 2024 was not represented "
        "as one monthly coverage record."
    )

december = december.iloc[0]

reference_strict_median = (
    reference[
        "median_households_at_least_90"
    ].median()
)

december_last_week = (
    december_daily.loc[
        december_daily[
            "reading_date"
        ] >= "2024-12-25"
    ]
)

december_earlier = (
    december_daily.loc[
        december_daily[
            "reading_date"
        ] < "2024-12-25"
    ]
)

december_checks = pd.DataFrame([
    {
        "criterion": (
            "Data reach 31 December 2024"
        ),
        "passed": (
            pd.Timestamp(
                december[
                    "last_observed_date"
                ]
            )
            == pd.Timestamp(
                "2024-12-31"
            )
        ),
    },
    {
        "criterion": (
            "All 31 December dates represented"
        ),
        "passed": (
            int(
                december[
                    "observed_calendar_dates"
                ]
            )
            == 31
        ),
    },
    {
        "criterion": (
            "December median strict-household coverage "
            "is at least 80% of May-November"
        ),
        "passed": (
            december[
                "median_households_at_least_90"
            ]
            >= 0.80
            * reference_strict_median
        ),
    },
    {
        "criterion": (
            "Final seven days retain at least 80% "
            "of earlier December strict coverage"
        ),
        "passed": (
            december_last_week[
                "households_at_least_90"
            ].median()
            >= 0.80
            * december_earlier[
                "households_at_least_90"
            ].median()
        ),
    },
])

INCLUDE_DECEMBER_2024 = bool(
    december_checks[
        "passed"
    ].all()
)

SMART15_CONTEXT_END_EXCLUSIVE = (
    pd.Timestamp("2025-01-01")
    if INCLUDE_DECEMBER_2024
    else pd.Timestamp("2024-12-01")
)

december_decision = pd.DataFrame([{
    "include_december_2024": (
        INCLUDE_DECEMBER_2024
    ),
    "smart15_context_start": (
        "2024-05-01"
    ),
    "smart15_context_end_inclusive": (
        (
            SMART15_CONTEXT_END_EXCLUSIVE
            - pd.Timedelta(days=1)
        ).date()
    ),
    "decision_rule": (
        "Include only if December is complete, "
        "not truncated and retains >=80% of "
        "post-recovery strict-household coverage."
    ),
}])

display(monthly_coverage)
display(december_daily)
display(december_checks)
display(december_decision)

monthly_coverage.to_csv(
    OUTPUT_DIR
    / "05_smart15_may_december_monthly_coverage.csv",
    index=False,
)
december_daily.to_csv(
    OUTPUT_DIR
    / "06_december_2024_daily_coverage.csv",
    index=False,
)
december_checks.to_csv(
    OUTPUT_DIR
    / "07_december_2024_inclusion_checks.csv",
    index=False,
)
december_decision.to_csv(
    OUTPUT_DIR
    / "08_december_2024_final_decision.csv",
    index=False,
)

In [ ]:
# ============================================================
# STEP 6 — SIX-HOUR INGESTION AND DAILY RECONSTRUCTION
# ============================================================

sixhour_files = sorted(
    PROJECT_ROOT.glob(
        SIXHOUR_PATTERN
    )
)

if len(sixhour_files) != 5:
    raise FileNotFoundError(
        "Exactly five smart_6hour_*.csv files "
        "are required in the project root."
    )

sixhour_parts = []

for path in sixhour_files:
    header = pd.read_csv(
        path,
        nrows=0,
    )
    header.columns = (
        header.columns
        .astype(str)
        .str.strip()
    )

    required_columns = [
        "household_ID",
        "BRANCH",
        "DATE",
        "TIME",
        "TOTAL_IMPORT (kWh)",
    ]

    missing = (
        set(required_columns)
        - set(header.columns)
    )

    if missing:
        raise KeyError(
            f"{path.name} is missing "
            f"{sorted(missing)}"
        )

    part = pd.read_csv(
        path,
        usecols=required_columns,
        low_memory=False,
    )

    part.columns = (
        part.columns
        .astype(str)
        .str.strip()
    )

    part["household_ID"] = clean_id(
        part["household_ID"]
    )

    part["reading_datetime"] = (
        parse_explicit_datetime(
            part,
            "DATE",
            "TIME",
        )
    )

    part["total_import_register_kwh"] = (
        clean_numeric(
            part[
                "TOTAL_IMPORT (kWh)"
            ]
        )
    )

    part["BRANCH"] = (
        part["BRANCH"]
        .map(canonical_branch)
    )

    part["_source_file"] = path.name

    part = part.dropna(
        subset=[
            "household_ID",
            "reading_datetime",
            "total_import_register_kwh",
        ]
    )

    sixhour_parts.append(
        part[
            [
                "household_ID",
                "BRANCH",
                "reading_datetime",
                "total_import_register_kwh",
                "_source_file",
            ]
        ]
    )

sixhour = pd.concat(
    sixhour_parts,
    ignore_index=True,
)

del sixhour_parts

branch_consistency = (
    sixhour.groupby(
        "household_ID"
    )["BRANCH"]
    .nunique(dropna=True)
)

branch_mode = (
    sixhour.dropna(
        subset=["BRANCH"]
    )
    .groupby(
        "household_ID"
    )["BRANCH"]
    .agg(
        lambda values:
            values.mode().iloc[0]
            if not values.mode().empty
            else np.nan
    )
    .rename("BRANCH")
    .reset_index()
)

branch_mode[
    "branch_conflict"
] = (
    branch_mode[
        "household_ID"
    ].map(
        branch_consistency
    ).fillna(0).gt(1)
)

sixhour = (
    sixhour.sort_values(
        [
            "household_ID",
            "reading_datetime",
            "_source_file",
        ]
    )
    .drop_duplicates(
        [
            "household_ID",
            "reading_datetime",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

sixhour["previous_datetime"] = (
    sixhour.groupby(
        "household_ID"
    )["reading_datetime"]
    .shift(1)
)

sixhour[
    "previous_register_kwh"
] = (
    sixhour.groupby(
        "household_ID"
    )[
        "total_import_register_kwh"
    ]
    .shift(1)
)

sixhour["gap_hours"] = (
    sixhour["reading_datetime"]
    - sixhour["previous_datetime"]
).dt.total_seconds() / 3600.0

sixhour["raw_delta_kwh"] = (
    sixhour[
        "total_import_register_kwh"
    ]
    - sixhour[
        "previous_register_kwh"
    ]
)

sixhour["rate_kwh_per_hour"] = (
    sixhour["raw_delta_kwh"]
    / sixhour["gap_hours"]
)

valid_interval = (
    sixhour["gap_hours"].gt(0)
    & sixhour[
        "gap_hours"
    ].le(
        SIXHOUR_MAX_GAP_HOURS
    )
    & sixhour[
        "raw_delta_kwh"
    ].ge(0)
    & sixhour[
        "rate_kwh_per_hour"
    ].le(
        MAX_RATE_KWH_PER_HOUR
    )
)

intervals = (
    sixhour.loc[
        valid_interval,
        [
            "household_ID",
            "previous_datetime",
            "reading_datetime",
            "gap_hours",
            "raw_delta_kwh",
        ],
    ]
    .copy()
)

intervals["interval_id"] = (
    np.arange(len(intervals))
)

intervals["previous_date"] = (
    intervals[
        "previous_datetime"
    ].dt.normalize()
)

intervals["current_date"] = (
    intervals[
        "reading_datetime"
    ].dt.normalize()
)

same_day = intervals.loc[
    intervals[
        "previous_date"
    ].eq(
        intervals[
            "current_date"
        ]
    )
].copy()

same_day["analysis_date"] = (
    same_day["previous_date"]
)
same_day["allocated_hours"] = (
    same_day["gap_hours"]
)
same_day["allocated_kwh"] = (
    same_day["raw_delta_kwh"]
)

cross_day = intervals.loc[
    (
        intervals[
            "current_date"
        ]
        - intervals[
            "previous_date"
        ]
    ).dt.days.eq(1)
].copy()

midnight = (
    cross_day["current_date"]
)

hours_before_midnight = (
    midnight
    - cross_day[
        "previous_datetime"
    ]
).dt.total_seconds() / 3600.0

hours_after_midnight = (
    cross_day[
        "reading_datetime"
    ]
    - midnight
).dt.total_seconds() / 3600.0

total_cross_hours = (
    hours_before_midnight
    + hours_after_midnight
)

before = cross_day.copy()
before["analysis_date"] = (
    before["previous_date"]
)
before["allocated_hours"] = (
    hours_before_midnight
)
before["allocated_kwh"] = (
    before["raw_delta_kwh"]
    * hours_before_midnight
    / total_cross_hours
)

after = cross_day.copy()
after["analysis_date"] = (
    after["current_date"]
)
after["allocated_hours"] = (
    hours_after_midnight
)
after["allocated_kwh"] = (
    after["raw_delta_kwh"]
    * hours_after_midnight
    / total_cross_hours
)

segments = pd.concat(
    [
        same_day[
            [
                "household_ID",
                "analysis_date",
                "allocated_hours",
                "allocated_kwh",
                "interval_id",
            ]
        ],
        before[
            [
                "household_ID",
                "analysis_date",
                "allocated_hours",
                "allocated_kwh",
                "interval_id",
            ]
        ],
        after[
            [
                "household_ID",
                "analysis_date",
                "allocated_hours",
                "allocated_kwh",
                "interval_id",
            ]
        ],
    ],
    ignore_index=True,
)

source_interval_total_kwh = (
    intervals.loc[
        intervals[
            "previous_date"
        ].eq(
            intervals[
                "current_date"
            ]
        )
        | (
            (
                intervals[
                    "current_date"
                ]
                - intervals[
                    "previous_date"
                ]
            ).dt.days.eq(1)
        ),
        "raw_delta_kwh",
    ].sum()
)

allocated_segment_total_kwh = (
    segments[
        "allocated_kwh"
    ].sum()
)

allocation_difference_kwh = (
    allocated_segment_total_kwh
    - source_interval_total_kwh
)

if abs(
    allocation_difference_kwh
) > 1e-6:
    raise RuntimeError(
        "Six-hour daily allocation failed energy-conservation check."
    )

sixhour_daily_all = (
    segments.groupby(
        [
            "household_ID",
            "analysis_date",
        ],
        as_index=False,
    )
    .agg(
        observed_kwh=(
            "allocated_kwh",
            "sum",
        ),
        covered_hours=(
            "allocated_hours",
            "sum",
        ),
        contributing_intervals=(
            "interval_id",
            "nunique",
        ),
    )
    .merge(
        branch_mode,
        on="household_ID",
        how="left",
        validate="many_to_one",
    )
)


# Households without a recognized branch do not appear in branch_mode.
# Treat those missing mappings as ineligible instead of applying boolean
# inversion to a floating-point column.
sixhour_daily_all[
    "branch_conflict"
] = (
    sixhour_daily_all[
        "branch_conflict"
    ]
    .astype("boolean")
    .fillna(True)
)

sixhour_daily_all[
    "estimated_24h_kwh"
] = (
    sixhour_daily_all[
        "observed_kwh"
    ]
    / sixhour_daily_all[
        "covered_hours"
    ].replace(
        0,
        np.nan,
    )
    * 24.0
)

sixhour_daily_primary = (
    sixhour_daily_all.loc[
        sixhour_daily_all[
            "covered_hours"
        ].ge(
            SIXHOUR_PRIMARY_MIN_COVERED_HOURS
        )
        & ~sixhour_daily_all[
            "branch_conflict"
        ]
        & sixhour_daily_all[
            "BRANCH"
        ].notna()
    ]
    .copy()
)

sixhour_daily_sensitivity = (
    sixhour_daily_all.loc[
        sixhour_daily_all[
            "covered_hours"
        ].ge(
            SIXHOUR_SENSITIVITY_MIN_COVERED_HOURS
        )
        & ~sixhour_daily_all[
            "branch_conflict"
        ]
        & sixhour_daily_all[
            "BRANCH"
        ].notna()
    ]
    .copy()
)

sixhour_daily_primary[
    "analysis_date"
] = pd.to_datetime(
    sixhour_daily_primary[
        "analysis_date"
    ]
)

sixhour_daily_sensitivity[
    "analysis_date"
] = pd.to_datetime(
    sixhour_daily_sensitivity[
        "analysis_date"
    ]
)

weather_mapping = (
    weather[
        [
            "branch",
            "weather_series_id",
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "branch": "BRANCH",
        }
    )
)

sixhour_daily_primary = (
    sixhour_daily_primary.merge(
        weather_mapping,
        on="BRANCH",
        how="left",
        validate="many_to_one",
    )
)

sixhour_daily_sensitivity = (
    sixhour_daily_sensitivity.merge(
        weather_mapping,
        on="BRANCH",
        how="left",
        validate="many_to_one",
    )
)


# Validate the 24-hour maximum-gap reconstruction against monthly billing
# before using it in contextual event analyses.
monthly_validation_source = pd.read_csv(
    MONTHLY_PATH,
    low_memory=False,
)
monthly_validation_source.columns = (
    monthly_validation_source.columns
    .astype(str)
    .str.strip()
)
monthly_validation_source["household_ID"] = clean_id(
    monthly_validation_source["household_ID"]
)
monthly_validation_source["month_date"] = parse_explicit_datetime(
    monthly_validation_source,
    "month",
)
monthly_validation_source["month_period"] = (
    monthly_validation_source["month_date"].dt.to_period("M")
)
monthly_validation_source["consumption_kwh"] = clean_numeric(
    monthly_validation_source["consumption"]
)
monthly_validation_source = (
    monthly_validation_source.dropna(
        subset=[
            "household_ID",
            "month_period",
            "consumption_kwh",
        ]
    )
    .loc[
        lambda frame:
            frame["consumption_kwh"].ge(0)
    ]
    .drop_duplicates(
        [
            "household_ID",
            "month_period",
        ],
        keep="last",
    )
    .copy()
)

sixhour_daily_all["month_period"] = (
    sixhour_daily_all["analysis_date"].dt.to_period("M")
)

sixhour_monthly_reconstructed = (
    sixhour_daily_all.groupby(
        [
            "household_ID",
            "month_period",
        ],
        as_index=False,
    )
    .agg(
        reconstructed_kwh=(
            "observed_kwh",
            "sum",
        ),
        covered_hours=(
            "covered_hours",
            "sum",
        ),
    )
)

sixhour_monthly_reconstructed["calendar_hours"] = (
    sixhour_monthly_reconstructed["month_period"]
    .dt.days_in_month
    * 24.0
)

sixhour_monthly_reconstructed["coverage_fraction"] = (
    sixhour_monthly_reconstructed["covered_hours"]
    / sixhour_monthly_reconstructed["calendar_hours"]
).clip(upper=1.0)

sixhour_monthly_validation_rows = (
    monthly_validation_source.merge(
        sixhour_monthly_reconstructed,
        on=[
            "household_ID",
            "month_period",
        ],
        how="inner",
        validate="one_to_one",
    )
    .loc[
        lambda frame:
            frame["coverage_fraction"].ge(
                SIXHOUR_MONTHLY_MIN_COVERAGE
            )
    ]
    .copy()
)

if len(sixhour_monthly_validation_rows) >= 10:
    (
        sixhour_validation_spearman,
        sixhour_validation_spearman_p,
    ) = spearmanr(
        sixhour_monthly_validation_rows[
            "consumption_kwh"
        ],
        sixhour_monthly_validation_rows[
            "reconstructed_kwh"
        ],
    )
else:
    sixhour_validation_spearman = np.nan
    sixhour_validation_spearman_p = np.nan

sixhour_validation_error = (
    sixhour_monthly_validation_rows[
        "reconstructed_kwh"
    ]
    - sixhour_monthly_validation_rows[
        "consumption_kwh"
    ]
)

sixhour_validation_ape = (
    sixhour_validation_error.abs()
    / sixhour_monthly_validation_rows[
        "consumption_kwh"
    ].replace(0, np.nan)
)

sixhour_monthly_validation = pd.DataFrame([{
    "max_gap_hours": SIXHOUR_MAX_GAP_HOURS,
    "minimum_daily_covered_hours": (
        SIXHOUR_PRIMARY_MIN_COVERED_HOURS
    ),
    "monthly_coverage_cutoff": (
        SIXHOUR_MONTHLY_MIN_COVERAGE
    ),
    "matched_household_months": len(
        sixhour_monthly_validation_rows
    ),
    "matched_households": (
        sixhour_monthly_validation_rows[
            "household_ID"
        ].nunique()
    ),
    "median_coverage_fraction": (
        sixhour_monthly_validation_rows[
            "coverage_fraction"
        ].median()
    ),
    "spearman_r": sixhour_validation_spearman,
    "spearman_p": sixhour_validation_spearman_p,
    "median_ape": sixhour_validation_ape.median(),
    "mean_bias_kwh": sixhour_validation_error.mean(),
}])

SIXHOUR_BILLING_VALIDATION_PASS = bool(
    sixhour_monthly_validation[
        "matched_household_months"
    ].iloc[0]
    >= SIXHOUR_VALIDATION_MIN_MATCHED_MONTHS
    and sixhour_monthly_validation[
        "spearman_r"
    ].iloc[0]
    >= SIXHOUR_VALIDATION_MIN_SPEARMAN
    and sixhour_monthly_validation[
        "median_ape"
    ].iloc[0]
    <= SIXHOUR_VALIDATION_MAX_MEDIAN_APE
    and abs(
        allocation_difference_kwh
    )
    <= 1e-6
)

if not SIXHOUR_BILLING_VALIDATION_PASS:
    raise RuntimeError(
        "The 24-hour maximum-gap six-hour reconstruction "
        "did not pass the prespecified billing-validation gate."
    )

sixhour_daily_audit = pd.DataFrame([{
    "raw_rows_after_deduplication": (
        len(sixhour)
    ),
    "households": (
        sixhour[
            "household_ID"
        ].nunique()
    ),
    "valid_intervals_gap_le_8h": (
        len(intervals)
    ),
    "primary_household_days_18h": (
        len(sixhour_daily_primary)
    ),
    "primary_households": (
        sixhour_daily_primary[
            "household_ID"
        ].nunique()
    ),
    "sensitivity_household_days_12h": (
        len(
            sixhour_daily_sensitivity
        )
    ),
    "date_start": (
        sixhour_daily_all[
            "analysis_date"
        ].min()
    ),
    "date_end": (
        sixhour_daily_all[
            "analysis_date"
        ].max()
    ),
    "maximum_gap_hours": SIXHOUR_MAX_GAP_HOURS,
    "billing_validation_pass": (
        SIXHOUR_BILLING_VALIDATION_PASS
    ),
    "source_interval_total_kwh": (
        source_interval_total_kwh
    ),
    "allocated_segment_total_kwh": (
        allocated_segment_total_kwh
    ),
    "allocation_difference_kwh": (
        allocation_difference_kwh
    ),
}])

display(sixhour_daily_audit)
display(sixhour_monthly_validation)

sixhour_daily_audit.to_csv(
    OUTPUT_DIR
    / "09_sixhour_daily_reconstruction_audit.csv",
    index=False,
)
sixhour_monthly_validation.to_csv(
    OUTPUT_DIR
    / "09A_sixhour_monthly_billing_validation.csv",
    index=False,
)
sixhour_daily_primary.to_parquet(
    OUTPUT_DIR
    / "10_sixhour_daily_primary_18h.parquet",
    index=False,
)
sixhour_daily_sensitivity.to_parquet(
    OUTPUT_DIR
    / "11_sixhour_daily_sensitivity_12h.parquet",
    index=False,
)

In [ ]:
# ============================================================
# STEP 7 — STRICT 15-MINUTE DAILY CONTEXTUAL COHORT
# ============================================================

connection = duckdb.connect(
    database=":memory:"
)

smart15_daily = connection.execute(f"""
SELECT
    household_ID,
    CAST(reading_date AS DATE)
        AS analysis_date,
    valid_import_intervals,
    observed_import_kwh
FROM read_parquet('{daily_pattern}')
WHERE reading_date >= DATE '2024-05-01'
  AND reading_date < DATE '{
      SMART15_CONTEXT_END_EXCLUSIVE:%Y-%m-%d
  }'
  AND valid_import_intervals >= {
      SMART15_STRICT_INTERVALS
  }
  AND observed_import_kwh IS NOT NULL
""").df()

connection.close()

smart15_daily["household_ID"] = (
    clean_id(
        smart15_daily[
            "household_ID"
        ]
    )
)

smart15_daily["analysis_date"] = (
    pd.to_datetime(
        smart15_daily[
            "analysis_date"
        ]
    )
)

smart15_household_counts = (
    smart15_daily.groupby(
        "household_ID",
        as_index=False,
    )
    .agg(
        strict_days=(
            "analysis_date",
            "nunique",
        )
    )
)

eligible_smart15_ids = set(
    smart15_household_counts.loc[
        smart15_household_counts[
            "strict_days"
        ].ge(
            SMART15_MIN_STRICT_DAYS
        ),
        "household_ID",
    ]
)

smart15_daily = smart15_daily.loc[
    smart15_daily[
        "household_ID"
    ].isin(
        eligible_smart15_ids
    )
].copy()

smart15_daily = (
    smart15_daily.merge(
        branch_mode,
        on="household_ID",
        how="left",
        validate="many_to_one",
    )
)

smart15_daily[
    "branch_conflict"
] = (
    smart15_daily[
        "branch_conflict"
    ]
    .astype("boolean")
    .fillna(True)
)

smart15_daily = smart15_daily.loc[
    ~smart15_daily[
        "branch_conflict"
    ]
    & smart15_daily[
        "BRANCH"
    ].notna()
].copy()

smart15_daily = smart15_daily.merge(
    weather_mapping,
    on="BRANCH",
    how="left",
    validate="many_to_one",
)

smart15_daily_audit = pd.DataFrame([{
    "context_start": (
        smart15_daily[
            "analysis_date"
        ].min()
    ),
    "context_end": (
        smart15_daily[
            "analysis_date"
        ].max()
    ),
    "december_included": (
        INCLUDE_DECEMBER_2024
    ),
    "strict_interval_requirement": (
        SMART15_STRICT_INTERVALS
    ),
    "minimum_strict_days_per_household": (
        SMART15_MIN_STRICT_DAYS
    ),
    "households": (
        smart15_daily[
            "household_ID"
        ].nunique()
    ),
    "household_days": len(
        smart15_daily
    ),
}])

display(smart15_daily_audit)

smart15_daily_audit.to_csv(
    OUTPUT_DIR
    / "12_smart15_contextual_cohort_audit.csv",
    index=False,
)
smart15_daily.to_parquet(
    OUTPUT_DIR
    / "13_smart15_strict_daily_contextual.parquet",
    index=False,
)

In [ ]:
# ============================================================
# STEP 8 — TARIFF REVISION EVENT STUDY
# ============================================================
# Primary: six-hour household-day estimates with >=18 covered hours.
# High-resolution check: strict 15-minute days for 16 July 2024.
#
# Date pairs are:
# - within 28 days before and after;
# - the same weekday;
# - non-holidays;
# - matched to minimize temperature and rainfall differences.
# ============================================================

holiday_dates = set(
    holidays["date"]
)

tariff_effect_rows = []
tariff_summary_rows = []
tariff_pair_rows = []

tariff_events = (
    tariffs.sort_values(
        "effective_start"
    )
    .copy()
)

for event in tariff_events.itertuples():
    event_date = pd.Timestamp(
        event.effective_start
    )

    pairs = (
        build_weather_matched_tariff_pairs(
            weather,
            event_date,
            holiday_dates,
            window_days=EVENT_WINDOW_DAYS,
        )
    )

    if not pairs.empty:
        pairs["event_id"] = (
            event.regime_id
        )
        tariff_pair_rows.append(
            pairs
        )

    effects, summary = evaluate_date_pairs(
        sixhour_daily_primary,
        pairs,
        "estimated_24h_kwh",
        event.regime_id,
        "sixhour_primary_gap24_day18_validated",
    )

    if not effects.empty:
        tariff_effect_rows.append(
            effects
        )

    summary[
        "effective_date"
    ] = event_date
    summary[
        "regime_name"
    ] = event.regime_name

    tariff_summary_rows.append(
        summary
    )

    if (
        event.regime_id
        == "tariff_2024_jul"
    ):
        effects_15, summary_15 = (
            evaluate_date_pairs(
                smart15_daily,
                pairs,
                "observed_import_kwh",
                event.regime_id,
                "smart15_strict_day",
            )
        )

        if not effects_15.empty:
            tariff_effect_rows.append(
                effects_15
            )

        summary_15[
            "effective_date"
        ] = event_date
        summary_15[
            "regime_name"
        ] = event.regime_name

        tariff_summary_rows.append(
            summary_15
        )

tariff_household_effects = (
    pd.concat(
        tariff_effect_rows,
        ignore_index=True,
    )
    if tariff_effect_rows
    else pd.DataFrame()
)

tariff_event_summary = (
    pd.concat(
        tariff_summary_rows,
        ignore_index=True,
    )
    if tariff_summary_rows
    else pd.DataFrame()
)

tariff_date_pairs = (
    pd.concat(
        tariff_pair_rows,
        ignore_index=True,
    )
    if tariff_pair_rows
    else pd.DataFrame()
)

if not tariff_event_summary.empty:
    tariff_event_summary[
        "holm_adjusted_p"
    ] = adjust_pvalues(
        tariff_event_summary[
            "p_value"
        ],
        method="holm",
    )

display(tariff_event_summary)

tariff_date_pairs.to_csv(
    OUTPUT_DIR
    / "14_tariff_weather_matched_date_pairs.csv",
    index=False,
)
tariff_household_effects.to_csv(
    OUTPUT_DIR
    / "15_tariff_household_pre_post_effects.csv",
    index=False,
)
tariff_event_summary.to_csv(
    OUTPUT_DIR
    / "16_tariff_revision_event_summary.csv",
    index=False,
)

In [ ]:
# ============================================================
# STEP 9 — MONTHLY BILLING TARIFF LABELS
# ============================================================
# Monthly billing is secondary because billing cycles can cross an
# effective date. Months containing a tariff transition are labelled
# mixed_tariff_regime and are not treated as pure pre/post months.
# ============================================================

monthly = pd.read_csv(
    MONTHLY_PATH,
    low_memory=False,
)

monthly.columns = (
    monthly.columns
    .astype(str)
    .str.strip()
)

monthly["household_ID"] = clean_id(
    monthly["household_ID"]
)
monthly["month_date"] = (
    parse_explicit_datetime(
        monthly,
        "month",
    )
)
monthly["month_period"] = (
    monthly[
        "month_date"
    ].dt.to_period("M")
)
monthly["consumption_kwh"] = (
    clean_numeric(
        monthly["consumption"]
    )
)

monthly = monthly.dropna(
    subset=[
        "household_ID",
        "month_period",
        "consumption_kwh",
    ]
).copy()

monthly = monthly.loc[
    monthly[
        "consumption_kwh"
    ].ge(0)
].copy()

unique_months = pd.DataFrame({
    "month_period": sorted(
        monthly[
            "month_period"
        ].unique()
    )
})

def label_calendar_month(period):
    month_start = period.start_time.normalize()
    month_end = period.end_time.normalize()

    overlapping = tariffs.loc[
        tariffs[
            "effective_start"
        ].le(month_end)
        & (
            tariffs[
                "effective_end"
            ].isna()
            | tariffs[
                "effective_end"
            ].ge(month_start)
        )
    ]

    if len(overlapping) == 1:
        row = overlapping.iloc[0]

        if (
            row["effective_start"]
            <= month_start
            and (
                pd.isna(
                    row["effective_end"]
                )
                or row[
                    "effective_end"
                ] >= month_end
            )
        ):
            return row["regime_id"]

    if overlapping.empty:
        return "outside_verified_tariff_range"

    return "mixed_tariff_regime"

unique_months[
    "tariff_month_label"
] = (
    unique_months[
        "month_period"
    ].map(
        label_calendar_month
    )
)

monthly = monthly.merge(
    unique_months,
    on="month_period",
    how="left",
    validate="many_to_one",
)

display(unique_months)

unique_months.to_csv(
    OUTPUT_DIR
    / "17_calendar_month_tariff_labels.csv",
    index=False,
)
monthly[
    [
        "household_ID",
        "month_period",
        "consumption_kwh",
        "tariff_month_label",
    ]
].to_csv(
    OUTPUT_DIR
    / "18_monthly_billing_with_tariff_labels.csv",
    index=False,
)

In [ ]:
# ============================================================
# STEP 10 — OFFICIAL HOLIDAY ANALYSIS
# ============================================================
# Each holiday is compared with nearby non-holiday controls that:
# - have the same weekday;
# - are inside the same implemented tariff regime;
# - are within 28 days;
# - are selected using temperature, rainfall and date proximity;
# - include at least two controls for each household.
# ============================================================

weather_unique = (
    weather.drop_duplicates(
        [
            "weather_series_id",
            "date",
        ]
    )
    .copy()
)

weather_unique[
    "tariff_regime"
] = assign_tariff_regime(
    weather_unique["date"],
    tariffs,
)

holiday_set = set(
    holidays["date"]
)

def build_holiday_controls(
    holiday_date,
    tariff_regime,
):
    holiday_date = pd.Timestamp(
        holiday_date
    )

    control_rows = []

    for series_id, series in (
        weather_unique.groupby(
            "weather_series_id"
        )
    ):
        holiday_weather = (
            series.loc[
                series["date"].eq(
                    holiday_date
                )
            ]
        )

        if holiday_weather.empty:
            continue

        holiday_weather = (
            holiday_weather.iloc[0]
        )

        candidates = series.loc[
            series["date"].between(
                holiday_date
                - pd.Timedelta(
                    days=EVENT_WINDOW_DAYS
                ),
                holiday_date
                + pd.Timedelta(
                    days=EVENT_WINDOW_DAYS
                ),
            )
            & ~series["date"].eq(
                holiday_date
            )
            & ~series["date"].isin(
                holiday_set
            )
            & series[
                "date"
            ].dt.dayofweek.eq(
                holiday_date.dayofweek
            )
            & series[
                "tariff_regime"
            ].eq(
                tariff_regime
            )
        ].copy()

        if candidates.empty:
            continue

        temperature_scale = max(
            float(
                candidates[
                    "temperature_mean_c"
                ].std(ddof=0)
            ),
            0.5,
        )
        rain_scale = max(
            float(
                np.log1p(
                    candidates[
                        "precipitation_corrected_mm_day"
                    ].clip(lower=0)
                ).std(ddof=0)
            ),
            0.25,
        )

        candidates[
            "weather_distance"
        ] = candidates.apply(
            lambda row:
                weather_distance(
                    holiday_weather[
                        "temperature_mean_c"
                    ],
                    holiday_weather[
                        "precipitation_corrected_mm_day"
                    ],
                    row[
                        "temperature_mean_c"
                    ],
                    row[
                        "precipitation_corrected_mm_day"
                    ],
                    temperature_scale,
                    rain_scale,
                ),
            axis=1,
        )

        candidates[
            "calendar_distance_days"
        ] = (
            candidates["date"]
            - holiday_date
        ).dt.days.abs()

        candidates["score"] = (
            candidates[
                "weather_distance"
            ]
            + candidates[
                "calendar_distance_days"
            ]
            / EVENT_WINDOW_DAYS
        )

        before = (
            candidates.loc[
                candidates[
                    "date"
                ].lt(
                    holiday_date
                )
            ]
            .nsmallest(
                MAX_HOLIDAY_CONTROLS // 2,
                "score",
            )
        )

        after = (
            candidates.loc[
                candidates[
                    "date"
                ].gt(
                    holiday_date
                )
            ]
            .nsmallest(
                MAX_HOLIDAY_CONTROLS // 2,
                "score",
            )
        )

        selected = pd.concat(
            [
                before,
                after,
            ],
            ignore_index=True,
        )

        if len(selected) < 2:
            selected = (
                candidates.nsmallest(
                    MAX_HOLIDAY_CONTROLS,
                    "score",
                )
            )

        for control in (
            selected.itertuples()
        ):
            control_rows.append({
                "weather_series_id": (
                    series_id
                ),
                "holiday_date": (
                    holiday_date
                ),
                "control_date": (
                    control.date
                ),
                "weather_distance": (
                    control.weather_distance
                ),
                "calendar_distance_days": (
                    control.calendar_distance_days
                ),
            })

    return pd.DataFrame(
        control_rows
    )


def evaluate_holiday(
    daily,
    outcome_column,
    source_name,
    holiday_row,
):
    holiday_date = (
        holiday_row.date
    )

    tariff_regime = (
        assign_tariff_regime(
            pd.Series(
                [holiday_date]
            ),
            tariffs,
        ).iloc[0]
    )

    if pd.isna(tariff_regime):
        return (
            pd.DataFrame(),
            pd.DataFrame([{
                "source": source_name,
                "holiday_date": holiday_date,
                "holiday_name": (
                    holiday_row.holiday_name
                ),
                "holiday_category": (
                    holiday_row.holiday_category
                ),
                "status": (
                    "excluded_before_first_verified_tariff_regime"
                ),
            }]),
            pd.DataFrame(),
        )

    controls = build_holiday_controls(
        holiday_date,
        tariff_regime,
    )

    if controls.empty:
        return (
            pd.DataFrame(),
            pd.DataFrame([{
                "source": source_name,
                "holiday_date": holiday_date,
                "holiday_name": (
                    holiday_row.holiday_name
                ),
                "holiday_category": (
                    holiday_row.holiday_category
                ),
                "status": (
                    "no_same_weekday_same_regime_controls"
                ),
            }]),
            controls,
        )

    daily_required = daily[
        [
            "household_ID",
            "analysis_date",
            "weather_series_id",
            outcome_column,
        ]
    ].dropna().copy()

    holiday_values = (
        daily_required.loc[
            daily_required[
                "analysis_date"
            ].eq(
                holiday_date
            )
        ]
        .rename(
            columns={
                outcome_column:
                    "holiday_kwh",
            }
        )
    )

    control_values = (
        controls.merge(
            daily_required,
            left_on=[
                "weather_series_id",
                "control_date",
            ],
            right_on=[
                "weather_series_id",
                "analysis_date",
            ],
            how="inner",
        )
    )

    control_household = (
        control_values.groupby(
            "household_ID",
            as_index=False,
        )
        .agg(
            valid_controls=(
                "control_date",
                "nunique",
            ),
            control_mean_kwh=(
                outcome_column,
                "mean",
            ),
        )
    )

    household_effects = (
        holiday_values[
            [
                "household_ID",
                "holiday_kwh",
            ]
        ]
        .merge(
            control_household,
            on="household_ID",
            how="inner",
            validate="one_to_one",
        )
    )

    household_effects = (
        household_effects.loc[
            household_effects[
                "valid_controls"
            ].ge(2)
        ]
        .copy()
    )

    household_effects[
        "difference_kwh"
    ] = (
        household_effects[
            "holiday_kwh"
        ]
        - household_effects[
            "control_mean_kwh"
        ]
    )

    household_effects[
        "percent_difference"
    ] = (
        household_effects[
            "difference_kwh"
        ]
        / household_effects[
            "control_mean_kwh"
        ].replace(
            0,
            np.nan,
        )
        * 100
    )

    statistic, p_value = (
        safe_wilcoxon(
            household_effects[
                "difference_kwh"
            ]
        )
    )

    status = (
        "completed"
        if len(household_effects)
        >= MIN_EVENT_HOUSEHOLDS
        else "insufficient_households"
    )

    summary = pd.DataFrame([{
        "source": source_name,
        "holiday_date": holiday_date,
        "holiday_name": (
            holiday_row.holiday_name
        ),
        "holiday_category": (
            holiday_row.holiday_category
        ),
        "tariff_regime": (
            tariff_regime
        ),
        "status": status,
        "households": len(
            household_effects
        ),
        "control_dates": (
            controls[
                "control_date"
            ].nunique()
        ),
        "median_holiday_kwh": (
            household_effects[
                "holiday_kwh"
            ].median()
        ),
        "median_control_kwh": (
            household_effects[
                "control_mean_kwh"
            ].median()
        ),
        "median_difference_kwh": (
            household_effects[
                "difference_kwh"
            ].median()
        ),
        "median_percent_difference": (
            household_effects[
                "percent_difference"
            ].median()
        ),
        "wilcoxon_statistic": (
            statistic
        ),
        "p_value": p_value,
        "rank_biserial": (
            rank_biserial_from_paired(
                household_effects[
                    "difference_kwh"
                ]
            )
        ),
        "interpretation": (
            "Holiday-associated consumption difference; "
            "not a causal effect."
        ),
    }])

    household_effects[
        "source"
    ] = source_name
    household_effects[
        "holiday_date"
    ] = holiday_date
    household_effects[
        "holiday_name"
    ] = holiday_row.holiday_name
    household_effects[
        "holiday_category"
    ] = holiday_row.holiday_category

    controls[
        "source"
    ] = source_name
    controls[
        "holiday_name"
    ] = holiday_row.holiday_name

    return (
        household_effects,
        summary,
        controls,
    )


holiday_effect_frames = []
holiday_summary_frames = []
holiday_control_frames = []

for holiday in holidays.itertuples():
    for (
        daily_data,
        outcome,
        source_name,
    ) in [
        (
            sixhour_daily_primary,
            "estimated_24h_kwh",
            "sixhour_primary_gap24_day18_validated",
        ),
        (
            smart15_daily,
            "observed_import_kwh",
            "smart15_strict_day",
        ),
    ]:
        min_date = (
            daily_data[
                "analysis_date"
            ].min()
        )
        max_date = (
            daily_data[
                "analysis_date"
            ].max()
        )

        if not (
            min_date
            <= holiday.date
            <= max_date
        ):
            continue

        effects, summary, controls = (
            evaluate_holiday(
                daily_data,
                outcome,
                source_name,
                holiday,
            )
        )

        if not effects.empty:
            holiday_effect_frames.append(
                effects
            )

        holiday_summary_frames.append(
            summary
        )

        if not controls.empty:
            holiday_control_frames.append(
                controls
            )

holiday_household_effects = (
    pd.concat(
        holiday_effect_frames,
        ignore_index=True,
    )
    if holiday_effect_frames
    else pd.DataFrame()
)

holiday_event_summary = (
    pd.concat(
        holiday_summary_frames,
        ignore_index=True,
    )
    if holiday_summary_frames
    else pd.DataFrame()
)

holiday_controls = (
    pd.concat(
        holiday_control_frames,
        ignore_index=True,
    )
    if holiday_control_frames
    else pd.DataFrame()
)

if not holiday_event_summary.empty:
    holiday_event_summary[
        "holm_adjusted_p_within_source"
    ] = np.nan

    for source_name, index in (
        holiday_event_summary.groupby(
            "source"
        ).groups.items()
    ):
        holiday_event_summary.loc[
            index,
            "holm_adjusted_p_within_source",
        ] = adjust_pvalues(
            holiday_event_summary.loc[
                index,
                "p_value",
            ],
            method="holm",
        )

category_rows = []

if not holiday_household_effects.empty:
    category_household = (
        holiday_household_effects.groupby(
            [
                "source",
                "holiday_category",
                "household_ID",
            ],
            as_index=False,
        )
        .agg(
            holidays_observed=(
                "holiday_date",
                "nunique",
            ),
            mean_difference_kwh=(
                "difference_kwh",
                "mean",
            ),
        )
    )

    for (
        source_name,
        category,
    ), group in category_household.groupby(
        [
            "source",
            "holiday_category",
        ]
    ):
        statistic, p_value = (
            safe_wilcoxon(
                group[
                    "mean_difference_kwh"
                ]
            )
        )

        category_rows.append({
            "source": source_name,
            "holiday_category": (
                category
            ),
            "households": len(group),
            "median_household_category_difference_kwh": (
                group[
                    "mean_difference_kwh"
                ].median()
            ),
            "wilcoxon_statistic": (
                statistic
            ),
            "p_value": p_value,
            "rank_biserial": (
                rank_biserial_from_paired(
                    group[
                        "mean_difference_kwh"
                    ]
                )
            ),
        })

holiday_category_summary = (
    pd.DataFrame(
        category_rows
    )
)

if not holiday_category_summary.empty:
    holiday_category_summary[
        "holm_adjusted_p"
    ] = adjust_pvalues(
        holiday_category_summary[
            "p_value"
        ],
        method="holm",
    )

display(holiday_event_summary)
display(holiday_category_summary)

holiday_controls.to_csv(
    OUTPUT_DIR
    / "19_holiday_control_date_selection.csv",
    index=False,
)
holiday_household_effects.to_csv(
    OUTPUT_DIR
    / "20_holiday_household_effects.csv",
    index=False,
)
holiday_event_summary.to_csv(
    OUTPUT_DIR
    / "21_holiday_event_summary.csv",
    index=False,
)
holiday_category_summary.to_csv(
    OUTPUT_DIR
    / "22_holiday_category_summary.csv",
    index=False,
)

In [ ]:
# ============================================================
# STEP 11 — WEATHER-CONSUMPTION ASSOCIATIONS
# ============================================================
# Unit of weather analysis:
#   weather_series_id × date
#
# Moratuwa, Nugegoda and Kelaniya returned an identical complete
# NASA POWER weather series in the executed audit. They are therefore
# combined into one regional weather observation for each date.
# Kalutara forms the second regional weather series.
#
# This avoids pretending that identical coarse-grid weather values
# are separate local meteorological observations.
# ============================================================

weather_analysis_columns = [
    "temperature_mean_c",
    "temperature_max_c",
    "temperature_min_c",
    "relative_humidity_percent",
    "precipitation_corrected_mm_day",
    "wind_speed_2m_m_s",
]

weather_series_daily = (
    weather[
        [
            "weather_series_id",
            "date",
            *weather_analysis_columns,
        ]
    ]
    .drop_duplicates(
        [
            "weather_series_id",
            "date",
        ]
    )
    .copy()
)

def weather_association_analysis(
    daily,
    outcome_column,
    source_name,
):
    series_date = (
        daily.groupby(
            [
                "weather_series_id",
                "analysis_date",
            ],
            as_index=False,
        )
        .agg(
            households=(
                "household_ID",
                "nunique",
            ),
            contributing_branches=(
                "BRANCH",
                "nunique",
            ),
            median_daily_kwh=(
                outcome_column,
                "median",
            ),
            mean_daily_kwh=(
                outcome_column,
                "mean",
            ),
        )
    )

    analysis = series_date.merge(
        weather_series_daily,
        left_on=[
            "weather_series_id",
            "analysis_date",
        ],
        right_on=[
            "weather_series_id",
            "date",
        ],
        how="inner",
        validate="one_to_one",
    )

    analysis[
        "tariff_regime"
    ] = assign_tariff_regime(
        analysis[
            "analysis_date"
        ],
        tariffs,
    )

    analysis["holiday_flag"] = (
        analysis[
            "analysis_date"
        ].isin(
            holiday_set
        ).astype(int)
    )

    analysis["weekday"] = (
        analysis[
            "analysis_date"
        ].dt.day_name()
    )

    analysis["month"] = (
        analysis[
            "analysis_date"
        ].dt.to_period("M").astype(str)
    )

    analysis[
        "log_precipitation"
    ] = np.log1p(
        analysis[
            "precipitation_corrected_mm_day"
        ].clip(lower=0)
    )

    analysis[
        "temperature_squared"
    ] = (
        analysis[
            "temperature_mean_c"
        ] ** 2
    )

    analysis[
        "cooling_degree_day_24"
    ] = (
        analysis[
            "temperature_mean_c"
        ] - 24.0
    ).clip(lower=0)

    correlation_rows = []

    for scope_id, scope_data in [
        ("pooled", analysis),
        *[
            (
                str(series_id),
                group.copy(),
            )
            for series_id, group
            in analysis.groupby(
                "weather_series_id"
            )
        ],
    ]:
        for variable in [
            "temperature_mean_c",
            "temperature_max_c",
            "temperature_min_c",
            "relative_humidity_percent",
            "precipitation_corrected_mm_day",
            "wind_speed_2m_m_s",
            "cooling_degree_day_24",
        ]:
            paired = scope_data[
                [
                    "median_daily_kwh",
                    variable,
                ]
            ].dropna()

            if len(paired) < 30:
                continue

            rho, p_value = spearmanr(
                paired[
                    "median_daily_kwh"
                ],
                paired[variable],
            )

            correlation_rows.append({
                "source": source_name,
                "weather_scope": scope_id,
                "variable": variable,
                "weather_series_dates": len(
                    paired
                ),
                "spearman_rho": rho,
                "p_value": p_value,
            })

    correlations = pd.DataFrame(
        correlation_rows
    )

    if not correlations.empty:
        correlations[
            "fdr_adjusted_p"
        ] = np.nan

        for scope_id, scope_index in (
            correlations.groupby(
                "weather_scope"
            ).groups.items()
        ):
            correlations.loc[
                scope_index,
                "fdr_adjusted_p",
            ] = adjust_pvalues(
                correlations.loc[
                    scope_index,
                    "p_value",
                ],
                method="fdr_bh",
            )

    model_data = analysis.dropna(
        subset=[
            "median_daily_kwh",
            "temperature_mean_c",
            "relative_humidity_percent",
            "log_precipitation",
            "wind_speed_2m_m_s",
            "tariff_regime",
        ]
    ).copy()

    model_data = model_data.sort_values(
        [
            "analysis_date",
            "weather_series_id",
        ]
    ).reset_index(drop=True)

    coefficient_table = pd.DataFrame()
    model_status = pd.DataFrame()

    if (
        len(model_data) >= 100
        and model_data[
            "analysis_date"
        ].nunique() >= 50
    ):
        formula = (
            "median_daily_kwh ~ "
            "temperature_mean_c + "
            "temperature_squared + "
            "log_precipitation + "
            "relative_humidity_percent + "
            "wind_speed_2m_m_s + "
            "holiday_flag + "
            "C(weather_series_id) + "
            "C(weekday) + "
            "C(month) + "
            "C(tariff_regime)"
        )

        try:
            model = smf.ols(
                formula=formula,
                data=model_data,
            ).fit(
                cov_type="cluster",
                cov_kwds={
                    "groups": (
                        model_data[
                            "analysis_date"
                        ]
                    )
                },
            )

            confidence = model.conf_int()

            coefficient_table = pd.DataFrame({
                "term": model.params.index,
                "coefficient": model.params.values,
                "standard_error_date_clustered": (
                    model.bse.values
                ),
                "p_value": model.pvalues.values,
                "ci_low": confidence[0].values,
                "ci_high": confidence[1].values,
                "source": source_name,
            })

            model_status = pd.DataFrame([{
                "source": source_name,
                "status": "completed",
                "weather_series_dates": len(
                    model_data
                ),
                "calendar_dates": (
                    model_data[
                        "analysis_date"
                    ].nunique()
                ),
                "weather_series": (
                    model_data[
                        "weather_series_id"
                    ].nunique()
                ),
                "r_squared": model.rsquared,
                "adjusted_r_squared": (
                    model.rsquared_adj
                ),
                "covariance": (
                    "date-clustered robust standard errors"
                ),
                "interpretation": (
                    "Associational weather-series-date model with "
                    "calendar, tariff and holiday controls. "
                    "Not a causal estimate."
                ),
            }])

        except Exception as error:
            model_status = pd.DataFrame([{
                "source": source_name,
                "status": "model_failed",
                "error": str(error),
            }])
    else:
        model_status = pd.DataFrame([{
            "source": source_name,
            "status": (
                "insufficient_weather_series_dates"
            ),
            "weather_series_dates": len(
                model_data
            ),
            "calendar_dates": (
                model_data[
                    "analysis_date"
                ].nunique()
            ),
        }])

    return (
        analysis,
        correlations,
        coefficient_table,
        model_status,
    )


weather_outputs = {}

for (
    daily_data,
    outcome,
    source_name,
) in [
    (
        sixhour_daily_primary,
        "estimated_24h_kwh",
        "sixhour_primary_gap24_day18_validated",
    ),
    (
        smart15_daily,
        "observed_import_kwh",
        "smart15_strict_day",
    ),
]:
    (
        series_date_data,
        correlations,
        coefficients,
        status,
    ) = weather_association_analysis(
        daily_data,
        outcome,
        source_name,
    )

    weather_outputs[
        source_name
    ] = {
        "data": series_date_data,
        "correlations": correlations,
        "coefficients": coefficients,
        "status": status,
    }

    series_date_data.to_csv(
        OUTPUT_DIR
        / f"23_{source_name}_weather_series_date.csv",
        index=False,
    )
    correlations.to_csv(
        OUTPUT_DIR
        / f"24_{source_name}_weather_correlations.csv",
        index=False,
    )
    coefficients.to_csv(
        OUTPUT_DIR
        / f"25_{source_name}_weather_model_coefficients.csv",
        index=False,
    )
    status.to_csv(
        OUTPUT_DIR
        / f"26_{source_name}_weather_model_status.csv",
        index=False,
    )

correlation_frames = [
    values[
        "correlations"
    ]
    for values in weather_outputs.values()
    if not values[
        "correlations"
    ].empty
]

weather_correlation_summary = (
    pd.concat(
        correlation_frames,
        ignore_index=True,
    )
    if correlation_frames
    else pd.DataFrame()
)

weather_model_status = pd.concat(
    [
        values["status"]
        for values in weather_outputs.values()
    ],
    ignore_index=True,
)

display(weather_correlation_summary)
display(weather_model_status)

weather_correlation_summary.to_csv(
    OUTPUT_DIR
    / "27_weather_correlation_summary.csv",
    index=False,
)
weather_model_status.to_csv(
    OUTPUT_DIR
    / "28_weather_model_status_summary.csv",
    index=False,
)

if not weather_outputs[
    "smart15_strict_day"
]["data"].empty:
    plot_data = weather_outputs[
        "smart15_strict_day"
    ]["data"]

    plt.figure(
        figsize=(8, 5)
    )
    plt.scatter(
        plot_data[
            "temperature_mean_c"
        ],
        plot_data[
            "median_daily_kwh"
        ],
        alpha=0.45,
        s=18,
    )
    plt.xlabel(
        "NASA POWER mean temperature (°C)"
    )
    plt.ylabel(
        "Weather-series-date median observed kWh"
    )
    plt.title(
        "Strict 15-minute consumption and temperature"
    )
    save_figure(
        "29_smart15_temperature_association.png"
    )

In [ ]:
# ============================================================
# STEP 12 — FINAL PROVENANCE, LIMITATIONS AND RELEASE GATE
# ============================================================

source_manifest = pd.read_csv(
    SOURCE_DIR
    / "source_manifest.csv"
)

limitations = pd.DataFrame([
    {
        "topic": "Causality",
        "limitation": (
            "All tariff, holiday and weather findings are "
            "associations. The design does not establish causality."
        ),
    },
    {
        "topic": "Six-hour daily reconstruction",
        "limitation": (
            "Daily values are standardized from cumulative-register "
            "intervals with at least 18 covered hours. Allocation "
            "across midnight assumes a constant average rate within "
            "each retained interval."
        ),
    },
    {
        "topic": "Six-hour contextual reconstruction",
        "limitation": (
            "Contextual household-day estimates use cumulative-register "
            "intervals up to 24 hours and require at least 18 covered "
            "hours per day. The threshold is retained only after "
            "prespecified monthly billing validation and assumes a "
            "constant average rate within each retained interval."
        ),
    },
    {
        "topic": "Strict 15-minute days",
        "limitation": (
            "A strict day contains at least 90 of 96 nominal "
            "intervals and can still miss up to 1.5 hours."
        ),
    },
    {
        "topic": "Weather spatial resolution",
        "limitation": (
            "NASA POWER meteorology is coarse. Branches with "
            "identical complete series are combined at the "
            "weather_series_id-date level rather than treated "
            "as distinct local weather observations."
        ),
    },
    {
        "topic": "Tariff transition",
        "limitation": (
            "Day-level analyses use actual effective dates. "
            "Calendar months containing a revision are labelled "
            "mixed_tariff_regime."
        ),
    },
    {
        "topic": "Holiday controls",
        "limitation": (
            "Holiday controls are same-weekday, same-regime and "
            "weather-proximate, but unobserved events may remain."
        ),
    },
    {
        "topic": "Independence from clustering",
        "limitation": (
            "No cluster label, archetype or clustering feature is "
            "used. Results must not be presented as cluster validation."
        ),
    },
])

limitations.to_csv(
    OUTPUT_DIR
    / "30_contextual_analysis_limitations.csv",
    index=False,
)

loaded_paths = [
    str(TARIFF_PATH),
    str(HOLIDAY_PATH),
    str(WEATHER_PATH),
    str(COORDINATE_PATH),
    str(MONTHLY_PATH),
    *[
        str(path)
        for path in sixhour_files
    ],
    *[
        str(path)
        for path in daily_files
    ],
]

cluster_path_violation = any(
    any(
        term in path.lower()
        for term in FORBIDDEN_CLUSTER_TERMS
    )
    for path in loaded_paths
)

expected_primary_tariff_events = set(
    tariffs["regime_id"]
)

completed_primary_tariff_events = set(
    tariff_event_summary.loc[
        tariff_event_summary[
            "source"
        ].eq(
            "sixhour_primary_gap24_day18_validated"
        )
        & tariff_event_summary[
            "status"
        ].eq(
            "completed"
        ),
        "event_id",
    ]
)

smart15_july_completed = (
    tariff_event_summary.loc[
        tariff_event_summary[
            "source"
        ].eq(
            "smart15_strict_day"
        )
        & tariff_event_summary[
            "event_id"
        ].eq(
            "tariff_2024_jul"
        )
        & tariff_event_summary[
            "status"
        ].eq(
            "completed"
        )
    ].shape[0]
    == 1
)

completed_holiday_sources = set(
    holiday_event_summary.loc[
        holiday_event_summary[
            "status"
        ].eq(
            "completed"
        ),
        "source",
    ]
)

completed_weather_sources = set(
    weather_model_status.loc[
        weather_model_status[
            "status"
        ].eq(
            "completed"
        ),
        "source",
    ]
)

release_checks = {
    "no_cluster_inputs_loaded": (
        not cluster_path_violation
    ),
    "tariffs_all_approved_implemented": (
        tariffs[
            "decision_status"
        ].eq(
            "approved_implemented"
        ).all()
    ),
    "tariff_sources_are_pucsl": (
        tariffs[
            "source_url"
        ].str.contains(
            "pucsl.gov.lk",
            case=False,
        ).all()
    ),
    "tariff_periods_exact_and_continuous": (
        set(actual_tariff_periods)
        == set(expected_tariff_periods)
    ),
    "holidays_all_public": (
        holidays[
            "is_public_holiday"
        ].all()
    ),
    "holiday_dates_unique": (
        not holidays[
            "date"
        ].duplicated().any()
    ),
    "nasa_weather_downloaded_complete": (
        not weather.empty
        and weather[
            "date"
        ].min()
        == pd.Timestamp(
            "2023-01-01"
        )
        and weather[
            "date"
        ].max()
        == pd.Timestamp(
            "2024-12-31"
        )
        and not weather[
            [
                "temperature_mean_c",
                "temperature_max_c",
                "temperature_min_c",
                "relative_humidity_percent",
                "precipitation_corrected_mm_day",
                "wind_speed_2m_m_s",
            ]
        ].isna().any().any()
    ),
    "december_decision_completed": (
        len(december_decision) == 1
    ),
    "sixhour_billing_validation_pass": (
        SIXHOUR_BILLING_VALIDATION_PASS
    ),
    "sixhour_daily_available": (
        not sixhour_daily_primary.empty
    ),
    "smart15_daily_available": (
        not smart15_daily.empty
    ),
    "all_five_primary_tariff_events_completed": (
        completed_primary_tariff_events
        == expected_primary_tariff_events
    ),
    "smart15_july_tariff_check_completed": (
        smart15_july_completed
    ),
    "holiday_analysis_completed_for_both_sources": (
        {
            "sixhour_primary_gap24_day18_validated",
            "smart15_strict_day",
        }.issubset(
            completed_holiday_sources
        )
    ),
    "weather_models_completed_for_both_sources": (
        {
            "sixhour_primary_gap24_day18_validated",
            "smart15_strict_day",
        }.issubset(
            completed_weather_sources
        )
    ),
}

release_gate = pd.DataFrame([
    {
        "check": check,
        "passed": bool(passed),
    }
    for check, passed
    in release_checks.items()
])

release_gate.to_csv(
    OUTPUT_DIR
    / "31_final_release_gate.csv",
    index=False,
)

hard_check_names = {
    "no_cluster_inputs_loaded",
    "tariffs_all_approved_implemented",
    "tariff_sources_are_pucsl",
    "tariff_periods_exact_and_continuous",
    "holidays_all_public",
    "holiday_dates_unique",
    "nasa_weather_downloaded_complete",
    "december_decision_completed",
    "sixhour_billing_validation_pass",
    "sixhour_daily_available",
    "smart15_daily_available",
    "smart15_july_tariff_check_completed",
}

hard_failed_checks = (
    release_gate.loc[
        ~release_gate["passed"]
        & release_gate["check"].isin(
            hard_check_names
        ),
        "check",
    ].tolist()
)

analytical_limitations = (
    release_gate.loc[
        ~release_gate["passed"]
        & ~release_gate["check"].isin(
            hard_check_names
        ),
        "check",
    ].tolist()
)

failed_checks = (
    hard_failed_checks
    + analytical_limitations
)

final_audit = {
    "analysis_name": (
        "Independent contextual tariff, holiday and weather checks"
    ),
    "generated_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "research_question": (
        "Were household electricity-consumption patterns "
        "associated with implemented tariff revisions, official "
        "public holidays, and weather conditions?"
    ),
    "independent_from_clustering": True,
    "december_2024_included": (
        INCLUDE_DECEMBER_2024
    ),
    "smart15_context_end_exclusive": (
        str(
            SMART15_CONTEXT_END_EXCLUSIVE.date()
        )
    ),
    "weather_unique_series": int(
        weather[
            "weather_series_id"
        ].nunique()
    ),
    "weather_analysis_unit": (
        "weather_series_id-date"
    ),
    "paired_effect_size": (
        "Wilcoxon matched-pairs rank-biserial correlation"
    ),
    "completed_primary_tariff_events": sorted(
        completed_primary_tariff_events
    ),
    "completed_holiday_sources": sorted(
        completed_holiday_sources
    ),
    "completed_weather_sources": sorted(
        completed_weather_sources
    ),
    "release_checks": (
        release_checks
    ),
    "execution_release_status": (
        "PASS"
        if not hard_failed_checks
        else "FAIL"
    ),
    "analytical_coverage_status": (
        "COMPLETE"
        if not analytical_limitations
        else "PASS_WITH_DATA_LIMITATIONS"
    ),
    "hard_failed_checks": hard_failed_checks,
    "analytical_limitations": analytical_limitations,
    "non_causal_interpretation_required": True,
}

with (
    OUTPUT_DIR
    / "FINAL_CONTEXTUAL_CHECKS_AUDIT.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        final_audit,
        handle,
        indent=2,
        default=str,
    )

output_files = sorted(
    [
        path
        for path in OUTPUT_DIR.rglob("*")
        if path.is_file()
        and path.name
        != "OUTPUT_INDEX.csv"
    ]
)

output_index = pd.DataFrame([
    {
        "relative_path": str(
            path.relative_to(
                OUTPUT_DIR
            )
        ),
        "size_bytes": (
            path.stat().st_size
        ),
        "file_type": (
            path.suffix.lower()
        ),
    }
    for path in output_files
])

output_index.to_csv(
    OUTPUT_DIR
    / "OUTPUT_INDEX.csv",
    index=False,
)

display(release_gate)
display(limitations)

if hard_failed_checks:
    raise RuntimeError(
        "Independent contextual-check execution gate failed: "
        + ", ".join(
            hard_failed_checks
        )
    )

print(
    "Independent contextual-check execution gate: PASS"
)
print(
    "Analytical coverage:",
    (
        "COMPLETE"
        if not analytical_limitations
        else "PASS WITH DATA LIMITATIONS"
    ),
)

if analytical_limitations:
    print(
        "Documented analytical limitations:",
        ", ".join(
            analytical_limitations
        ),
    )

print(
    "No cluster labels or archetypes were used."
)
print(
    "All interpretations must remain associational and non-causal."
)
